<a href="https://colab.research.google.com/github/jatinkumar-creator/machine-learning-portfolio/blob/main/07_Recommendation_Systems/notebooks/04_Item_Based_Collaborative_Filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 32: Item-Based Collaborative Filtering

## Objective
Build a movie recommendation system using item-based
collaborative filtering and the MovieLens dataset.

## Concepts
- Item-Based Collaborative Filtering
- Movie-User Matrix
- Cosine Similarity
- Similar Movie Discovery
- Rating Prediction
- Top-N Recommendations

## Libraries
Python, Pandas, NumPy, Matplotlib, Scikit-learn

In [1]:
#Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, mean_squared_error

import zipfile
import urllib.request
import os

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
#Download MovieLens
url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"

if not os.path.exists("ml-latest-small"):
    urllib.request.urlretrieve(url, "movielens.zip")

    with zipfile.ZipFile("movielens.zip", "r") as zip_ref:
        zip_ref.extractall(".")

ratings = pd.read_csv("ml-latest-small/ratings.csv")
movies = pd.read_csv("ml-latest-small/movies.csv")

print("Dataset loaded successfully.")
print("Ratings:", ratings.shape)
print("Movies:", movies.shape)

Dataset loaded successfully.
Ratings: (100836, 4)
Movies: (9742, 3)


In [3]:
#Train-test split
train_ratings, test_ratings = train_test_split(
    ratings,
    test_size=0.20,
    random_state=42
)

print("Training ratings:", train_ratings.shape)
print("Testing ratings:", test_ratings.shape)

Training ratings: (80668, 4)
Testing ratings: (20168, 4)


In [4]:
#Identify the most-rated movies
movie_counts = (
    train_ratings.groupby("movieId")
    .size()
    .sort_values(ascending=False)
)

top_movie_ids = movie_counts.head(500).index

train_top = train_ratings[
    train_ratings["movieId"].isin(top_movie_ids)
].copy()

test_top = test_ratings[
    test_ratings["movieId"].isin(top_movie_ids)
].copy()

print("Selected movies:", len(top_movie_ids))
print("Training interactions:", train_top.shape)
print("Testing interactions:", test_top.shape)

Selected movies: 500
Training interactions: (35076, 4)
Testing interactions: (8527, 4)


In [5]:
#Create the movie–user matrix
movie_user_matrix = train_top.pivot_table(
    index="movieId",
    columns="userId",
    values="rating"
)

movie_user_matrix_filled = movie_user_matrix.fillna(0)

print("Movie-user matrix shape:", movie_user_matrix.shape)

display(movie_user_matrix.head())

Movie-user matrix shape: (500, 608)


userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,NaN,NaN,NaN,NaN,4.5,NaN,NaN,NaN,...,NaN,NaN,4.0,3.0,4.0,2.5,NaN,2.5,3.0,5.0
2,NaN,NaN,NaN,NaN,NaN,4.0,NaN,4.0,NaN,NaN,...,NaN,4.0,NaN,5.0,3.5,NaN,NaN,2.0,NaN,NaN
3,4.0,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN
6,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,3.0,4.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2.5,NaN,NaN,NaN,NaN


In [6]:
#Calculate item similarity
item_similarity = cosine_similarity(
    movie_user_matrix_filled
)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=movie_user_matrix.index,
    columns=movie_user_matrix.index
)

print("Item similarity matrix shape:", item_similarity_df.shape)

display(item_similarity_df.iloc[:5, :5])

Item similarity matrix shape: (500, 500)


movieId,1,2,3,6,7
movieId,,,,,
1,1.000000,0.362258,0.191738,0.278680,0.184020
2,0.362258,1.000000,0.185543,0.228204,0.194008
3,0.191738,0.185543,1.000000,0.177748,0.276536
6,0.278680,0.228204,0.177748,1.000000,0.157961
7,0.184020,0.194008,0.276536,0.157961,1.000000


In [7]:
#Find similar movies
def find_similar_movies(movie_id, top_n=10):

    if movie_id not in item_similarity_df.index:
        print("Movie not found in the selected movie set.")
        return pd.DataFrame()

    similarity_scores = item_similarity_df[movie_id]

    # Remove the movie itself
    similarity_scores = similarity_scores.drop(
        index=movie_id
    )

    # Select the most similar movies
    top_similar = similarity_scores.nlargest(top_n)

    results = pd.DataFrame({
        "movieId": top_similar.index,
        "similarity": top_similar.values
    })

    results = results.merge(
        movies,
        on="movieId"
    )

    return results[
        ["title", "genres", "similarity"]
    ]

In [8]:
#Test similar movie discovery
# Select a movie from the training matrix
selected_movie_id = movie_user_matrix.index[0]

selected_title = movies.loc[
    movies["movieId"] == selected_movie_id,
    "title"
].iloc[0]

print("Selected movie:", selected_title)

similar_movies = find_similar_movies(
    selected_movie_id,
    top_n=10
)

display(similar_movies)

Selected movie: Toy Story (1995)


,title,genres,similarity
0,Back to the Future (1985),Adventure|Comedy|Sci-Fi,0.469494
1,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX,0.462791
2,"Shawshank Redemption, The (1994)",Crime|Drama,0.459610
3,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,0.458225
4,Forrest Gump (1994),Comedy|Drama|Romance|War,0.456527
5,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,0.456314
6,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.453333
7,Willy Wonka & the Chocolate Factory (1971),Children|Comedy|Fantasy|Musical,0.450877
8,Beauty and the Beast (1991),Animation|Children|Fantasy|Musical|Romance|IMAX,0.450046
9,Shrek (2001),Adventure|Animation|Children|Comedy|Fantasy|Ro...,0.449573


In [9]:
#Create the prediction function
def predict_item_rating(user_id, movie_id):

    # Use the training-set average as a fallback
    fallback = train_top["rating"].mean()

    if movie_id not in item_similarity_df.index:
        return fallback

    if user_id not in movie_user_matrix.columns:
        return fallback

    # Ratings given by this user
    user_ratings = movie_user_matrix[user_id].dropna()

    # Remove the target movie if it is already rated
    user_ratings = user_ratings.drop(
        index=movie_id,
        errors="ignore"
    )

    if user_ratings.empty:
        return fallback

    # Similarity between the target movie and rated movies
    similarities = item_similarity_df.loc[
        movie_id,
        user_ratings.index
    ]

    # Keep only positive similarities
    similarities = similarities[similarities > 0]

    if similarities.empty:
        return fallback

    # Keep corresponding ratings
    relevant_ratings = user_ratings.loc[
        similarities.index
    ]

    # Similarity-weighted average
    prediction = np.dot(
        similarities.values,
        relevant_ratings.values
    ) / similarities.values.sum()

    return np.clip(prediction, 0.5, 5.0)

In [10]:
#Test one prediction
sample = test_top.iloc[0]

sample_user = sample["userId"]
sample_movie = sample["movieId"]
actual_rating = sample["rating"]

predicted_rating = predict_item_rating(
    sample_user,
    sample_movie
)

sample_title = movies.loc[
    movies["movieId"] == sample_movie,
    "title"
].iloc[0]

print("User ID:", sample_user)
print("Movie:", sample_title)
print("Actual rating:", actual_rating)
print("Predicted rating:", round(predicted_rating, 2))

User ID: 288.0
Movie: In the Line of Fire (1993)
Actual rating: 3.0
Predicted rating: 3.48


In [11]:
#Build the recommendation function
def recommend_item_based(
    user_id,
    n_recommendations=10
):

    if user_id not in movie_user_matrix.columns:
        print("User not found in training data.")
        return pd.DataFrame()

    # Movies already rated by the user
    watched_movies = set(
        movie_user_matrix[user_id].dropna().index
    )

    # Candidate movies not rated by the user
    candidate_movies = [
        movie_id
        for movie_id in movie_user_matrix.index
        if movie_id not in watched_movies
    ]

    recommendations = []

    for movie_id in candidate_movies:

        predicted_rating = predict_item_rating(
            user_id,
            movie_id
        )

        recommendations.append({
            "movieId": movie_id,
            "predicted_rating": predicted_rating
        })

    recommendations_df = pd.DataFrame(
        recommendations
    )

    if recommendations_df.empty:
        return recommendations_df

    recommendations_df = recommendations_df.merge(
        movies,
        on="movieId"
    )

    recommendations_df = recommendations_df.sort_values(
        by="predicted_rating",
        ascending=False
    )

    return recommendations_df[
        ["title", "genres", "predicted_rating"]
    ].head(n_recommendations)

In [12]:
#Generate recommendations
sample_user_id = train_top["userId"].iloc[0]

print("Recommendations for user:", sample_user_id)

item_recommendations = recommend_item_based(
    sample_user_id,
    n_recommendations=10
)

display(item_recommendations)

Recommendations for user: 57


,title,genres,predicted_rating
71,North by Northwest (1959),Action|Adventure|Mystery|Romance|Thriller,3.887320
93,"Graduate, The (1967)",Comedy|Drama|Romance,3.873584
72,Some Like It Hot (1959),Comedy|Crime,3.843950
84,12 Angry Men (1957),Drama,3.829339
317,Inception (2010),Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX,3.828638
74,It's a Wonderful Life (1946),Children|Drama|Fantasy|Romance,3.827497
320,"King's Speech, The (2010)",Drama,3.819754
91,Annie Hall (1977),Comedy|Romance,3.818662
15,Taxi Driver (1976),Crime|Drama|Thriller,3.817030
90,Amadeus (1984),Drama,3.813860


In [13]:
#Evaluate the model
evaluation_sample = test_top.sample(
    n=min(2000, len(test_top)),
    random_state=42
)

actual_values = []
predicted_values = []

for row in evaluation_sample.itertuples(index=False):

    prediction = predict_item_rating(
        row.userId,
        row.movieId
    )

    actual_values.append(row.rating)
    predicted_values.append(prediction)

item_mae = mean_absolute_error(
    actual_values,
    predicted_values
)

item_rmse = np.sqrt(
    mean_squared_error(
        actual_values,
        predicted_values
    )
)

print("Item-Based MAE:", round(item_mae, 4))
print("Item-Based RMSE:", round(item_rmse, 4))

Item-Based MAE: 0.6782
Item-Based RMSE: 0.8754


In [14]:
#Compare with the average baseline
baseline_value = train_top["rating"].mean()

baseline_predictions = [
    baseline_value
] * len(actual_values)

baseline_mae = mean_absolute_error(
    actual_values,
    baseline_predictions
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        actual_values,
        baseline_predictions
    )
)

comparison = pd.DataFrame({
    "Model": [
        "Average Rating Baseline",
        "Item-Based Collaborative Filtering"
    ],
    "MAE": [
        baseline_mae,
        item_mae
    ],
    "RMSE": [
        baseline_rmse,
        item_rmse
    ]
})

display(comparison)

,Model,MAE,RMSE
0,Average Rating Baseline,0.783222,0.975799
1,Item-Based Collaborative Filtering,0.678214,0.875431
